# Lab 6 · Notebook 4 (advanced): The real survey file

**This notebook is optional.** It is for you if notebooks 1 to 3 felt comfortable and you want to see what working with real data is like. Nothing in the assessed work depends on it.

**Time:** about an hour, and it's fine to do it over two sessions.

The table you used in the lecture and notebooks 1 to 3 was a tidied copy. This notebook uses the **original file** that the researchers published with their paper (Nicholls *et al.*, *Plants, People, Planet*). It has 23 columns instead of 13, and a few of the problems you meet in every real dataset.

**What is different here.** The notebook is in two parts:

- **Part A, cleaning:** four problems to find and fix. For each one you decide what to do, and the notebook explains why that decision matters.
- **Part B, analysis:** three questions that go further than the lecture.

There are fewer demos than in notebooks 1 to 3: each challenge tells you what to find and gives you a hint, and the self-checks tell you if you got there. If you get stuck, look back at notebooks 1 to 3: every tool you need is in one of them, apart from the few new ones introduced here.

## The extra columns

Each row is still one survey. The new columns are:

| Column | What it means |
|---|---|
| `Year`, `Date` | when the survey happened. `Date` is written like `2017-08-16` |
| `Plot` | which plot on the allotment, usually named after the plot-holder (`Pete`, `Karen`) |
| `Community` | `Community` for a shared plot, `Individual` for one person's plot |
| `Crop_Generic` | the crop (called `Crop` in the tidy file) |
| `Plants` | how many plants were in the watched patch |
| `Total_Flowers` | how many open flowers were in the patch (called `Flowers` in the tidy file) |
| `Beetles`, `Flies`, `Butterflies`, `Solitary_Bees`, `Wasps` | visits by other insect groups |
| `Terrestris` … `Hortorum` | bumblebee visits, split into six species. Added together, they give `Bumblebees` |

In [ ]:
# Run this first. It loads the survey file exactly as the researchers published it.
import urllib.request
from pathlib import Path

import pandas as pd

FILE = "pollinators.csv"
URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/" + FILE

if not Path(FILE).exists():
    urllib.request.urlretrieve(URL, FILE)

raw = pd.read_csv(FILE)
print("Loaded", FILE, raw.shape)

# Part A: Cleaning

Cleaning doesn't mean "delete everything that looks odd". It means finding out what is odd, working out **why**, and then deciding what to do. Sometimes the right decision is to change nothing.

Work on a **copy**, so you can always go back to the original:

In [ ]:
# Make a working copy. Changes to df won't touch raw.
df = raw.copy()
df.info()

## Challenge A1: The column that won't work

Try to take out the `Hortorum` column with `df["Hortorum"]`. You will get a `KeyError`, even though `Hortorum` is listed in `info()` above.

The problem is invisible on screen: whoever typed the spreadsheet put a **space** after the name, so the column is really called `"Hortorum "`. To see it, print the names as a list, which puts quote marks around each one: `print(list(df.columns))`.

**Your task:**

1. Find the column name with the space in it.
2. Remove extra spaces from the start and end of **every** column name in one line. Column names have string methods too, through `.str`: `df.columns.str.strip()` gives back the names with the spaces stripped off. Store that back in `df.columns`.

In [ ]:
# TODO A1 - find and fix the column name


# --- self-check (do not edit) ---
assert "Hortorum" in df.columns, "There is still no column called exactly 'Hortorum'. Try df.columns = df.columns.str.strip()"
assert all(c == c.strip() for c in df.columns), "Some column names still start or end with a space"
print("Fixed: df['Hortorum'] works now, and so will every other column")

Why this matters: one stray space in a header can make a script fail on someone else's computer, or make you think a column is missing. Stripping every column name straight after loading a file is a good habit.

## Challenge A2: Missing values

**Your task:**

1. Count the missing values in each column with `df.isna().sum()`. `.isna()` makes a True/False table (True where a value is missing), and `.sum()` counts the Trues in each column. Which column has gaps?
2. Make a table `missing` with only the rows where that column is missing. A column has its own `.isna()`, which gives a mask.
3. Use `value_counts()` on the crops in `missing`. Store the crop with the most missing values in `most_missing`.

In [ ]:
# TODO A2 - where are the missing values?


# --- self-check (do not edit) ---
for name in ["missing", "most_missing"]:
    assert name in globals(), f"Make a variable called {name}"
assert len(missing) == 41, f"missing should have 41 rows, got {len(missing)}. Use df[df['Plants'].isna()]"
assert most_missing == "Blackberry", f"most_missing should be 'Blackberry', got {most_missing!r}"
print("Correct: 41 surveys have no plant count, and 27 of them are blackberry")

**Why blackberry?** Blackberry grows as a bramble: one tangle of stems where you can't tell where one plant ends and the next begins. The volunteers couldn't count the plants, so they left the cell empty. That's an honest gap, not a mistake.

**What to do about it?** It depends on the question.

- `df.dropna()` removes every row with a missing value anywhere. Here that would remove 41 surveys, including 42 bumblebee visits, and most of the blackberry data, just because of a column we don't need.
- We don't use `Plants` in any analysis (we use `Total_Flowers`), so the best decision is to **leave these rows in**.

If you did need `Plants`, you would remove the missing rows only for that calculation, and say so when you report the result.

**Check yourself:** run `len(df.dropna())` and work out how many surveys it would lose.

## Challenge A3: Rows that appear twice

`df.duplicated()` gives `True` for every row that is an exact copy of an earlier row.

**Your task:**

1. Store the number of repeated rows in `n_repeats`.
2. Look at some of them: `df[df.duplicated(keep=False)].head(10)`. (`keep=False` marks **every** copy, not just the second one, so you can see them side by side.) What do they have in common?

In [ ]:
# TODO A3 - repeated rows


# --- self-check (do not edit) ---
assert "n_repeats" in globals(), "Make a variable called n_repeats"
assert n_repeats == 33, f"n_repeats should be 33, got {n_repeats}. Try df.duplicated().sum()"
print("Correct: 33 rows are exact copies of an earlier row")

**Are they mistakes?** Look at the copies: the same plot, crop and date, with **no insects seen**. Of the 59 rows involved, 53 saw nothing at all.

Picture what happened: a volunteer watched several patches of runner bean on the same plot on the same afternoon, and nothing came. Each patch was a real survey, but because nothing was seen, nothing in the data tells the rows apart. So they look like copies, but they are real data.

What happens if you remove them anyway? `df.drop_duplicates()` would delete 33 real surveys, almost all of them zeros. The average number of bumblebees per survey would go **up** (from 0.338 to 0.343), and the crops would look busier than they really are.

**Decision: keep them.** Before you delete rows that look like duplicates, check whether the data could really produce identical rows.

**Check yourself:** compare `df["Bumblebees"].mean()` with `df.drop_duplicates()["Bumblebees"].mean()`.

## Challenge A4: Dates that are really text

Look at `df["Date"]` in `info()`: it is stored as text, not as a date. pandas can't tell that `2017-08-16` comes before `2017-09-01` while it is text.

`pd.to_datetime(df["Date"])` turns the text into real dates. A date column then has a set of tools under `.dt`, for example `.dt.month_name()` to get the month as a word, or `.dt.dayofyear`.

**Your task:**

1. Replace `df["Date"]` with the real dates.
2. The file also has a `Month` column, typed in by hand. Make a mask of the rows where `df["Date"].dt.month_name()` does **not** match `df["Month"]` (use `!=`), and store the rows in `mismatch`. Look at them.

In [ ]:
# TODO A4 - real dates, and a check against the Month column


# --- self-check (do not edit) ---
assert str(df["Date"].dtype).startswith("datetime"), "df['Date'] is still text. Use df['Date'] = pd.to_datetime(df['Date'])"
assert "mismatch" in globals(), "Make a variable called mismatch"
assert len(mismatch) == 6, f"mismatch should have 6 rows, got {len(mismatch)}"
print("Correct: 6 surveys were dated 1 September 2017 but labelled August")

All six are from Moulsecoomb, dated **1 September 2017** but labelled **August**. Probably the survey round started in August and finished a day late, and the month label was copied down. Either way, the date is the more reliable record, so make the `Month` column agree with it:

```python
df["Month"] = df["Date"].dt.month_name()
```

It only affects six rows, but you would never have found it without checking.

In [ ]:
# Run this to fix the six labels, and give two columns the same names as in the tidy file
df["Month"] = df["Date"].dt.month_name()
df = df.rename(columns={"Crop_Generic": "Crop", "Total_Flowers": "Flowers"})
print((df["Date"].dt.month_name() != df["Month"]).sum(), "mismatches left")

### What you did in Part A

| Problem | What you found | Decision |
|---|---|---|
| A space after a column name | `"Hortorum "` | strip every column name |
| Missing plant counts | 41, mostly brambles | keep the rows: we don't use `Plants` |
| Rows that appear twice | 33, almost all empty surveys | keep them: they are real zeros |
| Dates stored as text | 6 surveys with the wrong month | convert, and trust the date |

Two of the four decisions were "don't delete anything". That is common: often, the useful part of cleaning is the checking.

# Part B: Analysis

## Challenge B1: Do shared plots attract more insects?

Some plots are shared community gardens (`Community`), the rest are rented by one person (`Individual`). Is one kind of plot better for insects?

First, add a column with **all insect visits** in each survey. `axis=1` tells `.sum()` to add **across** each row instead of down each column:

```python
insects = ["Beetles", "Hoverflies", "Flies", "Butterflies", "Honeybees", "Solitary_Bees", "Wasps", "Bumblebees"]
df["All_insects"] = df[insects].sum(axis=1)
```

(The six bumblebee species are left out of the list, because they are already counted inside `Bumblebees`. Including them would count every bumblebee twice.)

**Your task:** make a table `plots`, grouped by `Community`, with three columns: the number of surveys, the total insect visits and the total flowers. Then add two columns: `per_survey` (insects ÷ surveys) and `per_flower` (insects ÷ flowers).

`.agg()` lets you make several summaries at once and name them:

```python
df.groupby("Community").agg(
    surveys=("All_insects", "size"),
    insects=("All_insects", "sum"),
    flowers=("Flowers", "sum"),
)
```

Each line reads: *new column name = (which column, which summary)*.

In [ ]:
# TODO B1 - community vs individual plots


# --- self-check (do not edit) ---
assert "plots" in globals(), "Make a variable called plots"
for col in ["per_survey", "per_flower"]:
    assert col in plots.columns, f"plots has no column called {col} yet"
assert round(plots.loc["Community", "per_survey"], 2) == 1.37, "Community plots should have about 1.37 insect visits per survey"
assert round(plots.loc["Individual", "per_flower"], 4) == 0.0142, "Individual plots should have about 0.0142 visits per flower"
print("Correct. Now compare the two columns: which kind of plot wins?")

**The answer flips.** Per survey, community plots win (1.37 against 1.08 insect visits). Per flower, individual plots win (0.0142 against 0.0102).

Both are true. Community plots tend to have bigger patches with more flowers, so a volunteer standing at one sees more insects. But each flower gets fewer visits. Which one you should report depends on the question: *"where will I see the most insects?"* is per survey; *"how well is each flower being visited?"* is per flower.

When the answer depends this much on what you divide by, report both, and say which question each one answers.

## Challenge B2: Which bumblebees visit broad bean?

In notebook 3 you found that broad bean depends almost entirely on bumblebees. But there are six species in the data, and they are different animals: their tongues range from short to very long, and that decides which flowers they can feed from.

**Your task:**

1. Make `species`: for each crop, the total visits by each of the six species. (Group by `Crop` and sum the six species columns.)
2. Keep only the crops with at least 20 bumblebee visits in total. `species.sum(axis=1)` adds across each row.
3. Turn the counts into **percentages of each crop's bumblebee visits**. Dividing a whole table by one number per row needs `.div(..., axis=0)`:

```python
species_pct = species.div(species.sum(axis=1), axis=0) * 100
```

Store the result, rounded to whole numbers, in `species_pct`.

In [ ]:
# TODO B2 - bumblebee species on each crop
sp = ["Terrestris", "Lapidarius", "Pascuorum", "Pratorum", "Hypnorum", "Hortorum"]



# --- self-check (do not edit) ---
assert "species_pct" in globals(), "Make a variable called species_pct"
assert len(species_pct) == 5, f"species_pct should have the 5 crops with at least 20 bumblebee visits, got {len(species_pct)}"
assert species_pct.loc["Bbean", "Pascuorum"] == 65, f"Broad bean should be 65% Pascuorum, got {species_pct.loc['Bbean', 'Pascuorum']}"
print("Correct. Read across the Bbean row: only two species visit broad bean at all")

Only two species visit broad bean in this data:

- ***Bombus pascuorum*** (the common carder bee), 65%. It has a fairly long tongue and can reach nectar at the bottom of the bean's long flower tube, pollinating the flower as it does.
- ***Bombus terrestris*** (the buff-tailed bumblebee), 35%. It has a **short** tongue and can't reach the nectar the normal way. It is well known for biting a hole in the side of bean flowers and drinking through it, which usually doesn't pollinate the flower at all. This is called **nectar robbing**.

So not every "bumblebee visit" to broad bean is the same, and the counts alone can't tell you which visits actually pollinated the flower. To find out, you would need a different kind of data, such as watching how each bee enters the flower.

Compare blackberry, where *B. terrestris* makes up two thirds of the visits. Blackberry flowers are open and shallow, so short tongues are no problem there.

## Challenge B3: Was 2018 a worse year?

Bumblebee visits went from 230 in 2017 to 318 in 2018. That looks like a rise, but 2018 had more than twice as many surveys.

**Your task:** make a table `years`, grouped by `Year`, with the number of surveys and the bumblebee and honeybee visits **per survey**. Store the 2018 bumblebee visits per survey, rounded to 2 decimal places, in `bb_2018`.

In [ ]:
# TODO B3 - per survey, by year


# --- self-check (do not edit) ---
assert "bb_2018" in globals(), "Make a variable called bb_2018"
assert bb_2018 == 0.28, f"bb_2018 should be 0.28, got {bb_2018}. Bumblebee visits in 2018 divided by the number of 2018 surveys"
print("Correct: 0.48 bumblebees per survey in 2017, 0.28 in 2018")

Per survey, bumblebee visits **fell** from 0.48 to 0.28, and honeybee visits fell too (0.52 to 0.36). The total went up only because more surveys were done in 2018.

Before you write "2018 was a bad year for bees", check whether the two years are comparable. 2017's surveys only started in late May, while 2018's ran from April. Are the same months, crops and allotments in both years? Try `df.groupby(["Year", "Month"]).size()` and see.

## Your own question

You now have every tool you need to ask your own question of this data. Some ideas:

- Do the bumblebee species peak in different months?
- Is there an allotment that does unusually well for insects, once you account for the number of surveys?
- Do solitary bees or hoverflies prefer different crops from bumblebees?

Write your question in a markdown cell first, then the code, then one or two sentences on what you found and how much you trust it.

In [ ]:
# Your question here